# 📈 Session 2: Reading the Futures Curve
## Commodities Club Seminar Series - Northeastern University

---

### Learning Objectives
1. **Visualize** how prices for different delivery dates form a term structure
2. **Interpret** upward vs downward slopes – what they indicate about market regime
3. **Connect** curve shape to fundamentals: inventory, supply/demand, hedging pressure
4. **Identify** how regimes change and what transitions mean for investors

### Key Concepts
- **Term Structure**: A snapshot of futures prices at various expiration dates
- **Contango**: Upward-sloping curve (near-term cheaper, far-term expensive)
- **Backwardation**: Downward-sloping curve (near-term expensive, far-term cheaper)
- **Convergence**: All futures prices must converge to spot at expiration

---
## Section 1: Setup and Data Download

In [ ]:
# ==============================================================================
# CELL 1: SSL FIX (Run this first if you encounter SSL certificate errors)
# ==============================================================================
import ssl
import os
import warnings

# SSL certificate workarounds for corporate/university networks
ssl._create_default_https_context = ssl._create_unverified_context
os.environ['PYTHONHTTPSVERIFY'] = '0'

try:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
except:
    pass

warnings.filterwarnings('ignore')
print("✅ SSL workarounds applied")

In [ ]:
# ==============================================================================
# CELL 2: Import Libraries
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from datetime import datetime, timedelta
import yfinance as yf

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print("✅ Libraries imported successfully")

In [ ]:
# ==============================================================================
# CELL 3: Download Real Market Data
# ==============================================================================
# We'll use multiple instruments to infer term structure:
# - CL=F: WTI Crude Front Month Futures
# - USO: US Oil Fund (rolls front-month contracts)
# - USL: US 12 Month Oil Fund (holds 12 months of contracts - flatter exposure)
# - BNO: Brent Oil Fund

start_date = '2007-01-01'
end_date = datetime.now().strftime('%Y-%m-%d')

print(f"Downloading data from {start_date} to {end_date}...")

# Batch download
tickers = ['CL=F', 'USO', 'USL', 'BNO', '^IRX']

try:
    raw = yf.download(tickers, start=start_date, end=end_date, progress=True)
    
    # Extract Close prices from MultiIndex
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw['Close'].copy()
    else:
        prices = raw[['Close']].copy()
        prices.columns = [tickers[0]]
    
    prices = prices.rename(columns={
        'CL=F': 'WTI_Front',
        'USO': 'USO',
        'USL': 'USL',
        'BNO': 'Brent',
        '^IRX': 'TBill'
    })
    
    prices = prices.dropna(how='all')
    USE_REAL_DATA = len(prices) > 100 and 'WTI_Front' in prices.columns
    
    if USE_REAL_DATA:
        print(f"\n✅ Downloaded {len(prices)} days of real market data")
        print(f"   Available columns: {list(prices.columns)}")
        print(f"   Date range: {prices.index[0].strftime('%Y-%m-%d')} to {prices.index[-1].strftime('%Y-%m-%d')}")
    else:
        print("⚠️ Insufficient data downloaded, will use synthetic data")
        
except Exception as e:
    print(f"⚠️ Download failed: {e}")
    print("   Will use synthetic data for demonstration")
    USE_REAL_DATA = False

In [ ]:
# ==============================================================================
# CELL 4: Generate Synthetic Data (Fallback or Supplement)
# ==============================================================================
# This creates realistic futures curve data based on historical patterns

def generate_futures_curve_data(start='2007-01-01', end='2025-01-01'):
    """
    Generate realistic multi-contract futures data with proper term structure dynamics.
    Models: Front month, 3-month, 6-month, 12-month contracts.
    """
    np.random.seed(42)
    dates = pd.date_range(start=start, end=end, freq='B')
    n = len(dates)
    
    # Define market regimes with realistic parameters
    regimes = [
        # (start_date, end_date, base_price, trend, vol, curve_slope, regime_name)
        ('2007-01-01', '2008-07-01', 60, 0.0015, 0.02, -0.02, 'Pre-Crisis Bull'),
        ('2008-07-01', '2009-03-01', 140, -0.004, 0.04, 0.08, 'Financial Crisis'),
        ('2009-03-01', '2011-01-01', 35, 0.002, 0.025, 0.12, 'Recovery Contango'),
        ('2011-01-01', '2014-06-01', 95, 0.0001, 0.015, 0.04, 'High Price Era'),
        ('2014-06-01', '2016-02-01', 105, -0.003, 0.03, 0.15, 'Oil Glut'),
        ('2016-02-01', '2018-10-01', 28, 0.001, 0.02, 0.06, 'Recovery'),
        ('2018-10-01', '2020-03-01', 70, -0.0005, 0.02, 0.03, 'Pre-COVID'),
        ('2020-03-01', '2020-05-01', 60, -0.02, 0.08, 0.40, 'COVID Crash'),
        ('2020-05-01', '2022-06-01', 20, 0.003, 0.025, -0.08, 'Post-COVID Backwardation'),
        ('2022-06-01', '2025-01-01', 120, -0.0008, 0.02, 0.02, 'Recent Period'),
    ]
    
    # Initialize price arrays for different tenors
    front_month = np.zeros(n)
    month_3 = np.zeros(n)
    month_6 = np.zeros(n)
    month_12 = np.zeros(n)
    
    current_price = 60
    
    for i, date in enumerate(dates):
        # Find current regime
        for start_d, end_d, base, trend, vol, slope, name in regimes:
            if pd.Timestamp(start_d) <= date < pd.Timestamp(end_d):
                break
        
        # Random walk with regime parameters
        shock = np.random.normal(trend, vol)
        current_price = current_price * (1 + shock)
        current_price = max(5, min(current_price, 200))  # Bounds
        
        # Generate term structure based on curve slope
        # Positive slope = contango, negative slope = backwardation
        # Add some noise to the slope
        daily_slope = slope + np.random.normal(0, abs(slope) * 0.3)
        
        front_month[i] = current_price
        month_3[i] = current_price * (1 + daily_slope * 0.25)  # 3 months
        month_6[i] = current_price * (1 + daily_slope * 0.50)  # 6 months
        month_12[i] = current_price * (1 + daily_slope * 1.00)  # 12 months
    
    df = pd.DataFrame({
        'Front_Month': front_month,
        'Month_3': month_3,
        'Month_6': month_6,
        'Month_12': month_12
    }, index=dates)
    
    # Smooth the data slightly
    for col in df.columns:
        df[col] = df[col].rolling(3, min_periods=1).mean()
    
    return df

# Generate synthetic futures curve data
synthetic_curves = generate_futures_curve_data()

if not USE_REAL_DATA:
    print("\n📊 Using synthetic futures curve data")
    print(f"   Generated {len(synthetic_curves)} trading days")
    print(f"   Contracts: {list(synthetic_curves.columns)}")
else:
    print("\n📊 Synthetic data generated as supplement")

---
## Section 2: What is the Futures Curve?

The **term structure** or **futures curve** is a snapshot of futures prices for a commodity at various expiration dates. Plotting price against delivery date reveals the market's current state.

In [ ]:
# ==============================================================================
# CELL 5: Visualize Sample Futures Curves (Contango vs Backwardation)
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sample dates representing different regimes
tenors = ['Spot', '1M', '3M', '6M', '12M']
tenor_months = [0, 1, 3, 6, 12]

# Contango example (e.g., 2015 oil glut)
contango_prices = [45, 47, 51, 56, 62]
ax1 = axes[0]
ax1.plot(tenor_months, contango_prices, 'o-', color='#E74C3C', linewidth=3, markersize=12)
ax1.fill_between(tenor_months, contango_prices, alpha=0.3, color='#E74C3C')
ax1.set_xlabel('Months to Expiration', fontsize=12)
ax1.set_ylabel('Futures Price ($)', fontsize=12)
ax1.set_title('CONTANGO\n(Upward Sloping Curve)', fontsize=14, fontweight='bold', color='#E74C3C')
ax1.set_xticks(tenor_months)
ax1.set_xticklabels(tenors)
ax1.annotate('Near-term\nCHEAPER', xy=(0, 45), xytext=(2, 42),
             fontsize=10, ha='center', arrowprops=dict(arrowstyle='->', color='gray'))
ax1.annotate('Far-term\nEXPENSIVE', xy=(12, 62), xytext=(10, 65),
             fontsize=10, ha='center', arrowprops=dict(arrowstyle='->', color='gray'))
ax1.text(6, 48, 'NEGATIVE\nRoll Yield', fontsize=11, ha='center', 
         bbox=dict(boxstyle='round', facecolor='#FADBD8', edgecolor='#E74C3C'))

# Backwardation example (e.g., 2022 supply crisis)
backwardation_prices = [110, 105, 95, 88, 82]
ax2 = axes[1]
ax2.plot(tenor_months, backwardation_prices, 'o-', color='#27AE60', linewidth=3, markersize=12)
ax2.fill_between(tenor_months, backwardation_prices, alpha=0.3, color='#27AE60')
ax2.set_xlabel('Months to Expiration', fontsize=12)
ax2.set_ylabel('Futures Price ($)', fontsize=12)
ax2.set_title('BACKWARDATION\n(Downward Sloping Curve)', fontsize=14, fontweight='bold', color='#27AE60')
ax2.set_xticks(tenor_months)
ax2.set_xticklabels(tenors)
ax2.annotate('Near-term\nEXPENSIVE', xy=(0, 110), xytext=(2, 115),
             fontsize=10, ha='center', arrowprops=dict(arrowstyle='->', color='gray'))
ax2.annotate('Far-term\nCHEAPER', xy=(12, 82), xytext=(10, 78),
             fontsize=10, ha='center', arrowprops=dict(arrowstyle='->', color='gray'))
ax2.text(6, 100, 'POSITIVE\nRoll Yield', fontsize=11, ha='center',
         bbox=dict(boxstyle='round', facecolor='#D5F5E3', edgecolor='#27AE60'))

plt.tight_layout()
plt.suptitle('The Two Shapes of the Futures Curve', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('curve_shapes.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n📖 KEY INSIGHT:")
print("   • Contango: You PAY more for future delivery → Negative roll yield when rolling contracts")
print("   • Backwardation: Future delivery is CHEAPER → Positive roll yield when rolling contracts")

---
## Section 3: Historical Term Structure Evolution

In [ ]:
# ==============================================================================
# CELL 6: Calculate Term Structure Metrics
# ==============================================================================

# Use synthetic data for full curve analysis
df = synthetic_curves.copy()

# Calculate curve slope (annualized)
# Slope = (12-month price / Front month price - 1) * 100
df['Curve_Slope'] = ((df['Month_12'] / df['Front_Month']) - 1) * 100

# Calculate front-to-3-month spread
df['Near_Spread'] = ((df['Month_3'] / df['Front_Month']) - 1) * 100

# Identify regime
df['Regime'] = np.where(df['Curve_Slope'] > 0, 'Contango', 'Backwardation')

# Annualized roll yield (negative of slope)
df['Roll_Yield_Ann'] = -df['Curve_Slope']

print("Term Structure Metrics Summary:")
print("="*60)
print(f"Average Curve Slope: {df['Curve_Slope'].mean():.2f}%")
print(f"Contango Days: {(df['Regime'] == 'Contango').sum()} ({(df['Regime'] == 'Contango').mean()*100:.1f}%)")
print(f"Backwardation Days: {(df['Regime'] == 'Backwardation').sum()} ({(df['Regime'] == 'Backwardation').mean()*100:.1f}%)")
print(f"\nCurve Slope Range: {df['Curve_Slope'].min():.1f}% to {df['Curve_Slope'].max():.1f}%")

In [ ]:
# ==============================================================================
# CELL 7: Visualize Term Structure Over Time
# ==============================================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Define crisis periods for highlighting
crisis_periods = [
    ('2008-07-01', '2009-03-01', 'Financial Crisis', '#ffcccc'),
    ('2014-06-01', '2016-02-01', 'Oil Glut', '#fff3cd'),
    ('2020-03-01', '2020-06-01', 'COVID', '#e1d5f0'),
]

# Panel 1: Price levels across tenors
ax1 = axes[0]
ax1.plot(df.index, df['Front_Month'], label='Front Month', color='black', linewidth=1.5)
ax1.plot(df.index, df['Month_3'], label='3-Month', color='#3498DB', linewidth=1, alpha=0.8)
ax1.plot(df.index, df['Month_6'], label='6-Month', color='#E67E22', linewidth=1, alpha=0.8)
ax1.plot(df.index, df['Month_12'], label='12-Month', color='#9B59B6', linewidth=1, alpha=0.8)

for start, end, label, color in crisis_periods:
    ax1.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.3, color=color)

ax1.set_ylabel('Price ($)', fontsize=12)
ax1.set_title('WTI Crude Oil Futures Prices by Tenor', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', ncol=4)
ax1.set_ylim(0, None)

# Panel 2: Curve Slope (Contango/Backwardation measure)
ax2 = axes[1]
colors = np.where(df['Curve_Slope'] > 0, '#E74C3C', '#27AE60')
ax2.fill_between(df.index, 0, df['Curve_Slope'], 
                  where=df['Curve_Slope'] > 0, color='#E74C3C', alpha=0.6, label='Contango')
ax2.fill_between(df.index, 0, df['Curve_Slope'],
                  where=df['Curve_Slope'] <= 0, color='#27AE60', alpha=0.6, label='Backwardation')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)

for start, end, label, color in crisis_periods:
    ax2.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.3, color=color)

ax2.set_ylabel('Curve Slope (%)', fontsize=12)
ax2.set_title('Term Structure Slope (12M vs Front Month)', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right')
ax2.set_ylim(-30, 50)

# Panel 3: Implied Roll Yield
ax3 = axes[2]
ax3.fill_between(df.index, 0, df['Roll_Yield_Ann'],
                  where=df['Roll_Yield_Ann'] > 0, color='#27AE60', alpha=0.6, label='Positive Roll')
ax3.fill_between(df.index, 0, df['Roll_Yield_Ann'],
                  where=df['Roll_Yield_Ann'] <= 0, color='#E74C3C', alpha=0.6, label='Negative Roll')
ax3.axhline(y=0, color='black', linestyle='-', linewidth=1)

for start, end, label, color in crisis_periods:
    ax2.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.3, color=color)

ax3.set_ylabel('Annualized Roll Yield (%)', fontsize=12)
ax3.set_xlabel('Date', fontsize=12)
ax3.set_title('Implied Roll Yield from Term Structure', fontsize=14, fontweight='bold')
ax3.legend(loc='upper right')
ax3.set_ylim(-50, 30)

plt.tight_layout()
plt.savefig('term_structure_evolution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n📖 OBSERVATIONS:")
print("   • Curve slope varies dramatically with market conditions")
print("   • Contango (red) tends to dominate during oversupply periods")
print("   • Backwardation (green) emerges during supply crunches")

---
## Section 4: Animated Futures Curve Through Time

Let's visualize how the shape of the curve changes across different market regimes.

In [ ]:
# ==============================================================================
# CELL 8: Snapshot Curves at Different Points in Time
# ==============================================================================

# Select representative dates for different market conditions
snapshot_dates = [
    ('2008-06-01', 'Pre-Crisis Peak (Jun 2008)', '#2C3E50'),
    ('2009-01-15', 'Financial Crisis (Jan 2009)', '#E74C3C'),
    ('2011-04-01', 'Recovery Period (Apr 2011)', '#3498DB'),
    ('2015-01-15', 'Oil Glut (Jan 2015)', '#E67E22'),
    ('2020-04-20', 'COVID Crash (Apr 2020)', '#9B59B6'),
    ('2022-03-01', 'Ukraine Crisis (Mar 2022)', '#27AE60'),
]

fig, ax = plt.subplots(figsize=(12, 7))

tenors = [0, 3, 6, 12]
tenor_labels = ['Front', '3M', '6M', '12M']

for date_str, label, color in snapshot_dates:
    try:
        date = pd.Timestamp(date_str)
        # Find nearest date in data
        idx = df.index.get_indexer([date], method='nearest')[0]
        row = df.iloc[idx]
        
        curve = [row['Front_Month'], row['Month_3'], row['Month_6'], row['Month_12']]
        ax.plot(tenors, curve, 'o-', label=label, color=color, linewidth=2.5, markersize=10)
    except:
        continue

ax.set_xlabel('Months to Expiration', fontsize=12)
ax.set_ylabel('Futures Price ($)', fontsize=12)
ax.set_title('Futures Curve Snapshots: How Shape Changes Over Time', fontsize=14, fontweight='bold')
ax.set_xticks(tenors)
ax.set_xticklabels(tenor_labels)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

# Add annotations
ax.annotate('Steep Backwardation\n(Supply Crisis)', xy=(8, 125), fontsize=10,
            bbox=dict(boxstyle='round', facecolor='#D5F5E3', alpha=0.8))
ax.annotate('Steep Contango\n(COVID Storage Crisis)', xy=(8, 35), fontsize=10,
            bbox=dict(boxstyle='round', facecolor='#FADBD8', alpha=0.8))

plt.tight_layout()
plt.savefig('curve_snapshots.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n📖 KEY OBSERVATIONS:")
print("   • COVID crash created extreme contango (front month collapsed)")
print("   • Supply crises create backwardation (near-term premium)")
print("   • Curve shape predicts roll yield for futures investors")

---
## Section 5: What Drives Curve Shapes?

In [ ]:
# ==============================================================================
# CELL 9: Drivers of Curve Shape - Conceptual Framework
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Contango Drivers
ax1 = axes[0]
contango_drivers = [
    'High Inventory\n(Oversupply)', 
    'Storage Costs\n(Carry Costs)',
    'Bullish Future\nExpectations',
    'Low Convenience\nYield'
]
contango_values = [35, 25, 25, 15]
colors_c = ['#E74C3C', '#C0392B', '#A93226', '#922B21']
wedges, texts, autotexts = ax1.pie(contango_values, labels=contango_drivers, autopct='%1.0f%%',
                                    colors=colors_c, startangle=90, explode=(0.05, 0, 0, 0))
ax1.set_title('CONTANGO DRIVERS\n(Upward Sloping Curve)', fontsize=14, fontweight='bold', color='#E74C3C')

# Backwardation Drivers
ax2 = axes[1]
backwardation_drivers = [
    'Low Inventory\n(Shortage)', 
    'Strong Current\nDemand',
    'Bearish Future\nOutlook',
    'High Convenience\nYield'
]
backwardation_values = [35, 25, 20, 20]
colors_b = ['#27AE60', '#229954', '#1E8449', '#196F3D']
wedges, texts, autotexts = ax2.pie(backwardation_values, labels=backwardation_drivers, autopct='%1.0f%%',
                                    colors=colors_b, startangle=90, explode=(0.05, 0, 0, 0))
ax2.set_title('BACKWARDATION DRIVERS\n(Downward Sloping Curve)', fontsize=14, fontweight='bold', color='#27AE60')

plt.suptitle('What Drives Futures Curve Shape?', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('curve_drivers.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n" + "="*70)
print("RULE OF THUMB:")
print("="*70)
print("   📦 Low Inventory  →  Backwardation (near-term scarcity premium)")
print("   📦 High Inventory →  Contango (storage costs push up far prices)")
print("="*70)

In [ ]:
# ==============================================================================
# CELL 10: Simulated Inventory vs Curve Slope Relationship
# ==============================================================================

# Create synthetic inventory proxy (inverse relationship with backwardation)
np.random.seed(123)
df['Inventory_Proxy'] = 50 + df['Curve_Slope'] * 1.2 + np.random.normal(0, 5, len(df))
df['Inventory_Proxy'] = df['Inventory_Proxy'].rolling(20).mean()  # Smooth

fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot curve slope
color1 = '#2C3E50'
ax1.fill_between(df.index, 0, df['Curve_Slope'],
                  where=df['Curve_Slope'] > 0, color='#E74C3C', alpha=0.4, label='Contango')
ax1.fill_between(df.index, 0, df['Curve_Slope'],
                  where=df['Curve_Slope'] <= 0, color='#27AE60', alpha=0.4, label='Backwardation')
ax1.axhline(y=0, color='black', linewidth=1)
ax1.set_ylabel('Curve Slope (%)', fontsize=12, color=color1)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(-30, 50)

# Plot inventory on secondary axis
ax2 = ax1.twinx()
color2 = '#3498DB'
ax2.plot(df.index, df['Inventory_Proxy'], color=color2, linewidth=2, label='Inventory Proxy')
ax2.set_ylabel('Inventory Level (Index)', fontsize=12, color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

# Add correlation annotation
corr = df['Curve_Slope'].corr(df['Inventory_Proxy'])
ax1.text(0.02, 0.98, f'Correlation: {corr:.2f}', transform=ax1.transAxes,
         fontsize=12, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax1.set_xlabel('Date', fontsize=12)
ax1.set_title('Inventory Levels Drive Curve Shape', fontsize=14, fontweight='bold')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('inventory_curve_relationship.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n📖 KEY INSIGHT:")
print(f"   Correlation between inventory and curve slope: {corr:.2f}")
print("   → High inventory → Contango (positive slope)")
print("   → Low inventory → Backwardation (negative slope)")

---
## Section 6: Regime Analysis & Statistics

In [ ]:
# ==============================================================================
# CELL 11: Regime Statistics
# ==============================================================================

# Regime breakdown by year
df['Year'] = df.index.year
yearly_regime = df.groupby('Year').agg({
    'Curve_Slope': 'mean',
    'Roll_Yield_Ann': 'mean',
    'Front_Month': 'mean',
    'Regime': lambda x: (x == 'Contango').mean() * 100
}).round(2)
yearly_regime.columns = ['Avg Slope (%)', 'Avg Roll Yield (%)', 'Avg Price ($)', 'Contango %']

print("="*70)
print("YEARLY TERM STRUCTURE ANALYSIS")
print("="*70)
print(yearly_regime.to_string())
print("="*70)

In [ ]:
# ==============================================================================
# CELL 12: Visualize Regime Statistics
# ==============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Average curve slope by year
ax1 = axes[0, 0]
colors = ['#E74C3C' if x > 0 else '#27AE60' for x in yearly_regime['Avg Slope (%)']]
bars = ax1.bar(yearly_regime.index, yearly_regime['Avg Slope (%)'], color=colors, alpha=0.8)
ax1.axhline(y=0, color='black', linewidth=1)
ax1.set_xlabel('Year')
ax1.set_ylabel('Average Curve Slope (%)')
ax1.set_title('Average Term Structure Slope by Year', fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Panel 2: Contango percentage by year
ax2 = axes[0, 1]
ax2.bar(yearly_regime.index, yearly_regime['Contango %'], color='#E74C3C', alpha=0.7, label='Contango')
ax2.bar(yearly_regime.index, 100 - yearly_regime['Contango %'], 
        bottom=yearly_regime['Contango %'], color='#27AE60', alpha=0.7, label='Backwardation')
ax2.axhline(y=50, color='black', linestyle='--', linewidth=1)
ax2.set_xlabel('Year')
ax2.set_ylabel('Percentage of Days (%)')
ax2.set_title('Market Regime Distribution by Year', fontweight='bold')
ax2.legend()
ax2.tick_params(axis='x', rotation=45)

# Panel 3: Implied roll yield by year
ax3 = axes[1, 0]
colors = ['#27AE60' if x > 0 else '#E74C3C' for x in yearly_regime['Avg Roll Yield (%)']]
ax3.bar(yearly_regime.index, yearly_regime['Avg Roll Yield (%)'], color=colors, alpha=0.8)
ax3.axhline(y=0, color='black', linewidth=1)
ax3.set_xlabel('Year')
ax3.set_ylabel('Average Roll Yield (%)')
ax3.set_title('Implied Annual Roll Yield by Year', fontweight='bold')
ax3.tick_params(axis='x', rotation=45)

# Panel 4: Price vs Curve Slope scatter
ax4 = axes[1, 1]
scatter = ax4.scatter(df['Front_Month'], df['Curve_Slope'], 
                       c=df['Roll_Yield_Ann'], cmap='RdYlGn', alpha=0.3, s=5)
ax4.axhline(y=0, color='black', linewidth=1)
ax4.set_xlabel('Front Month Price ($)')
ax4.set_ylabel('Curve Slope (%)')
ax4.set_title('Price Level vs Term Structure', fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax4)
cbar.set_label('Roll Yield (%)')

plt.tight_layout()
plt.savefig('regime_statistics.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## Section 7: Convergence to Spot

In [ ]:
# ==============================================================================
# CELL 13: Demonstrate Convergence Principle
# ==============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

# Create synthetic convergence example
days_to_expiry = np.arange(90, -1, -1)  # 90 days to expiry
spot_price = 75  # Spot stays relatively stable

# Contango scenario: futures start higher, converge down
contango_futures = spot_price + 5 * np.exp(-0.02 * (90 - days_to_expiry))
contango_futures[-1] = spot_price  # Converge at expiry

# Backwardation scenario: futures start lower, converge up
backwardation_futures = spot_price - 5 * np.exp(-0.02 * (90 - days_to_expiry))
backwardation_futures[-1] = spot_price  # Converge at expiry

ax.plot(days_to_expiry, [spot_price] * len(days_to_expiry), 'k--', 
        linewidth=2, label='Spot Price')
ax.plot(days_to_expiry, contango_futures, color='#E74C3C', 
        linewidth=2.5, label='Futures (Contango)')
ax.plot(days_to_expiry, backwardation_futures, color='#27AE60', 
        linewidth=2.5, label='Futures (Backwardation)')

ax.axvline(x=0, color='gray', linestyle=':', linewidth=1)
ax.annotate('EXPIRY\n(Convergence)', xy=(0, spot_price), xytext=(15, spot_price + 5),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='gray'))

# Highlight the convergence
ax.scatter([0], [spot_price], color='black', s=100, zorder=5, marker='*')

ax.set_xlabel('Days to Expiration', fontsize=12)
ax.set_ylabel('Price ($)', fontsize=12)
ax.set_title('Futures Convergence to Spot at Expiration', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.invert_xaxis()  # Days countdown
ax.grid(True, alpha=0.3)

# Add text boxes
ax.text(60, 79, 'Contango: Futures\nDRIFT DOWN\nto Spot', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='#FADBD8', alpha=0.8))
ax.text(60, 71, 'Backwardation: Futures\nDRIFT UP\nto Spot', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='#D5F5E3', alpha=0.8))

plt.tight_layout()
plt.savefig('convergence.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n📖 CONVERGENCE PRINCIPLE:")
print("   All futures contracts MUST converge to spot price at expiration.")
print("   If they didn't, risk-free arbitrage would be possible.")
print("\n   • Contango: Futures price drifts DOWN toward spot → Negative roll return")
print("   • Backwardation: Futures price drifts UP toward spot → Positive roll return")

---
## Section 8: Real Data Analysis (If Available)

In [ ]:
# ==============================================================================
# CELL 14: Real Data - USO vs USL Spread as Term Structure Proxy
# ==============================================================================

if USE_REAL_DATA and 'USO' in prices.columns and 'USL' in prices.columns:
    print("\n📊 REAL DATA: Analyzing USO vs USL Spread")
    print("   USO: Front-month roller (concentrated curve exposure)")
    print("   USL: 12-month roller (spread across the curve)\n")
    
    # Calculate normalized performance
    real_df = prices[['USO', 'USL']].dropna()
    
    if len(real_df) > 100:
        real_df['USO_Norm'] = real_df['USO'] / real_df['USO'].iloc[0] * 100
        real_df['USL_Norm'] = real_df['USL'] / real_df['USL'].iloc[0] * 100
        
        # The gap between USO and USL reflects term structure impact
        real_df['Spread'] = real_df['USO_Norm'] - real_df['USL_Norm']
        
        fig, axes = plt.subplots(2, 1, figsize=(14, 10))
        
        # Panel 1: Normalized performance
        ax1 = axes[0]
        ax1.plot(real_df.index, real_df['USO_Norm'], label='USO (Front Month)', color='#E74C3C', linewidth=1.5)
        ax1.plot(real_df.index, real_df['USL_Norm'], label='USL (12-Month Spread)', color='#3498DB', linewidth=1.5)
        ax1.axhline(y=100, color='black', linestyle='--', alpha=0.5)
        ax1.fill_between(real_df.index, real_df['USO_Norm'], real_df['USL_Norm'], 
                         alpha=0.3, color='gray', label='Term Structure Gap')
        ax1.set_ylabel('Normalized Price (Start = 100)')
        ax1.set_title('USO vs USL: Term Structure Impact on Returns', fontsize=14, fontweight='bold')
        ax1.legend(loc='upper right')
        
        # Panel 2: Spread
        ax2 = axes[1]
        ax2.fill_between(real_df.index, 0, real_df['Spread'],
                         where=real_df['Spread'] > 0, color='#27AE60', alpha=0.6, label='USO Outperforming')
        ax2.fill_between(real_df.index, 0, real_df['Spread'],
                         where=real_df['Spread'] <= 0, color='#E74C3C', alpha=0.6, label='USL Outperforming')
        ax2.axhline(y=0, color='black', linewidth=1)
        ax2.set_ylabel('USO - USL Spread')
        ax2.set_xlabel('Date')
        ax2.set_title('Performance Spread: Front Month vs Curve Exposure', fontsize=14, fontweight='bold')
        ax2.legend(loc='upper right')
        
        plt.tight_layout()
        plt.savefig('uso_usl_spread.png', dpi=150, bbox_inches='tight', facecolor='white')
        plt.show()
        
        # Summary statistics
        print(f"\n📊 Performance Summary:")
        print(f"   USO Total Return: {(real_df['USO_Norm'].iloc[-1] - 100):.1f}%")
        print(f"   USL Total Return: {(real_df['USL_Norm'].iloc[-1] - 100):.1f}%")
        print(f"   Term Structure Gap: {real_df['Spread'].iloc[-1]:.1f} points")
    else:
        print("   Insufficient overlapping data for USO/USL comparison")
else:
    print("\n📊 USO/USL data not available - using synthetic analysis only")

---
## Section 9: Key Takeaways Summary

In [ ]:
# ==============================================================================
# CELL 15: Summary Dashboard
# ==============================================================================

fig = plt.figure(figsize=(16, 10))

# Create grid
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# Panel 1: Contango vs Backwardation pie
ax1 = fig.add_subplot(gs[0, 0])
contango_pct = (df['Regime'] == 'Contango').mean() * 100
backwardation_pct = 100 - contango_pct
ax1.pie([contango_pct, backwardation_pct], 
        labels=[f'Contango\n{contango_pct:.1f}%', f'Backwardation\n{backwardation_pct:.1f}%'],
        colors=['#E74C3C', '#27AE60'], autopct='', startangle=90,
        explode=(0.05, 0.05))
ax1.set_title('Regime Distribution\n(Full Period)', fontweight='bold')

# Panel 2: Average roll yield by regime
ax2 = fig.add_subplot(gs[0, 1])
regime_stats = df.groupby('Regime')['Roll_Yield_Ann'].mean()
colors = ['#27AE60' if r == 'Backwardation' else '#E74C3C' for r in regime_stats.index]
bars = ax2.bar(regime_stats.index, regime_stats.values, color=colors, alpha=0.8)
ax2.axhline(y=0, color='black', linewidth=1)
ax2.set_ylabel('Average Roll Yield (%)')
ax2.set_title('Roll Yield by Regime', fontweight='bold')
for bar, val in zip(bars, regime_stats.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

# Panel 3: Curve slope distribution
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(df['Curve_Slope'], bins=50, color='#3498DB', alpha=0.7, edgecolor='white')
ax3.axvline(x=0, color='black', linewidth=2, linestyle='--')
ax3.axvline(x=df['Curve_Slope'].mean(), color='#E74C3C', linewidth=2, label=f'Mean: {df["Curve_Slope"].mean():.1f}%')
ax3.set_xlabel('Curve Slope (%)')
ax3.set_ylabel('Frequency')
ax3.set_title('Distribution of Curve Slopes', fontweight='bold')
ax3.legend()

# Panel 4-5: Key takeaways text
ax4 = fig.add_subplot(gs[1, :])
ax4.axis('off')

takeaways = """
╔══════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                    KEY TAKEAWAYS FROM SESSION 2                                      ║
╠══════════════════════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                                      ║
║  1. TERM STRUCTURE = futures prices plotted by expiration date                                       ║
║     → Upward slope = Contango | Downward slope = Backwardation                                       ║
║                                                                                                      ║
║  2. CONTANGO (most common): Near-term cheap, far-term expensive                                      ║
║     → Driven by: storage costs, high inventory, bullish expectations                                 ║
║     → Result: NEGATIVE roll yield (you pay more for each new contract)                               ║
║                                                                                                      ║
║  3. BACKWARDATION (supply crunches): Near-term expensive, far-term cheap                             ║
║     → Driven by: shortages, low inventory, strong current demand                                     ║
║     → Result: POSITIVE roll yield (you get paid via the structure)                                   ║
║                                                                                                      ║
║  4. CONVERGENCE: All futures must equal spot at expiration                                           ║
║     → This is why curve shape determines roll return                                                 ║
║                                                                                                      ║
║  5. INVENTORY IS KEY: Low inventory → Backwardation | High inventory → Contango                      ║
║     → Monitor inventory reports to anticipate regime changes                                         ║
║                                                                                                      ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════╝
"""

ax4.text(0.5, 0.5, takeaways, transform=ax4.transAxes, fontsize=11,
         verticalalignment='center', horizontalalignment='center',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#F8F9F9', edgecolor='#2C3E50', linewidth=2))

plt.suptitle('Session 2 Summary: Reading the Futures Curve', fontsize=16, fontweight='bold', y=0.98)
plt.savefig('session2_summary.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## 💬 Discussion Questions

1. **If you expect oil inventories to build significantly over the next quarter, what would you expect to happen to the futures curve? How would this affect a long USO position?**

2. **A news headline reads: "OPEC announces surprise production cut." What would you expect to happen to the WTI futures curve in the short term? Why?**

3. **Why might contango be considered the "normal" state for many commodities, especially those with significant storage costs?**

4. **During the COVID crash in April 2020, WTI futures briefly went negative. What does this tell you about the extreme contango that existed? What physical market conditions caused this?**

5. **If you're building a commodity allocation and want to minimize roll yield drag, what strategies might you consider based on today's analysis?**

---

### Next Session Preview: **Carry in Commodities**
- Understanding the carry trade in commodity futures
- Risk transfer and hedging pressure
- Building carry-based strategies